# Advanced Topics in Logistic Regression

## Overview
This notebook explores advanced concepts in logistic regression that are crucial for handling complex real-world problems.

## Key Topics
1. Multiclass Classification (Softmax Regression)
2. Regularization Techniques (L1, L2)
3. Feature Engineering
4. Handling Imbalanced Data
5. Model Interpretation

Let's dive into these advanced concepts with practical implementations.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import SMOTE

# Set random seed for reproducibility
np.random.seed(42)

## 1. Multiclass Classification (Softmax Regression)

Logistic regression can be extended to handle multiple classes using the softmax function, also known as multinomial logistic regression.

The softmax function for class $k$ out of $K$ classes is:
$$P(y=k|X) = \frac{e^{w_k^TX + b_k}}{\sum_{j=1}^{K} e^{w_j^TX + b_j}}$$

Let's implement multiclass classification with a synthetic dataset.

In [ ]:
# Generate synthetic dataset for multiclass classification
X, y = make_blobs(n_samples=1000, centers=3, n_features=2, random_state=42)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train multiclass logistic regression
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Print classification report
print('Classification Report for Multiclass:')
print(classification_report(y_test, y_pred))

# Plot decision boundaries
def plot_decision_boundary(X, y, model, scaler):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
    grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    Z = model.predict(grid)
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.4)
    plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.8)
    plt.title('Multiclass Decision Boundaries')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

plt.figure(figsize=(10, 8))
plot_decision_boundary(X_train, y_train, model, scaler)
plt.show()

## 2. Regularization Techniques (L1, L2)

Regularization helps prevent overfitting by adding a penalty term to the cost function.
- L2 regularization (Ridge): Adds squared magnitude of coefficients as penalty
- L1 regularization (Lasso): Adds absolute value of coefficients as penalty, can lead to feature selection

The cost function with regularization becomes:
$$J(w,b) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} log(\hat{y}^{(i)}) + (1-y^{(i)}) log(1 - \hat{y}^{(i)})] + \frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2$$ for L2

Let's compare models with different regularization techniques.

In [ ]:
# Generate synthetic dataset with more features
X, y = make_classification(n_samples=1000, n_features=20, n_redundant=10, random_state=42)

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models with different regularization
model_no_reg = LogisticRegression(penalty='none', solver='lbfgs', random_state=42)
model_l2 = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', random_state=42)
model_l1 = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', random_state=42)

model_no_reg.fit(X_train_scaled, y_train)
model_l2.fit(X_train_scaled, y_train)
model_l1.fit(X_train_scaled, y_train)

# Compare coefficients
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.bar(range(len(model_no_reg.coef_[0])), model_no_reg.coef_[0])
plt.title('No Regularization')
plt.ylabel('Coefficient Value')

plt.subplot(1, 3, 2)
plt.bar(range(len(model_l2.coef_[0])), model_l2.coef_[0])
plt.title('L2 Regularization')

plt.subplot(1, 3, 3)
plt.bar(range(len(model_l1.coef_[0])), model_l1.coef_[0])
plt.title('L1 Regularization')

plt.tight_layout()
plt.show()

# Compare performance
print('Accuracy without regularization:', model_no_reg.score(X_test_scaled, y_test))
print('Accuracy with L2 regularization:', model_l2.score(X_test_scaled, y_test))
print('Accuracy with L1 regularization:', model_l1.score(X_test_scaled, y_test))
print('Number of non-zero coefficients (L1):', np.sum(model_l1.coef_ != 0))

## 3. Feature Engineering

Feature engineering can significantly improve model performance by creating more informative features. Common techniques include:
- Polynomial features
- Interaction terms
- Binning continuous variables
- Encoding categorical variables

In [ ]:
# Generate non-linearly separable data
X, y = make_classification(n_samples=1000, n_features=2, n_classes=2, n_clusters_per_class=1, random_state=42)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Scale features
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

# Train models
model_basic = LogisticRegression(random_state=42)
model_poly = LogisticRegression(random_state=42)

model_basic.fit(scaler.fit_transform(X_train), y_train)
model_poly.fit(X_train_poly_scaled, y_train)

# Compare performance
print('Accuracy with original features:', model_basic.score(scaler.transform(X_test), y_test))
print('Accuracy with polynomial features:', model_poly.score(X_test_poly_scaled, y_test))

## 4. Handling Imbalanced Data

Real-world datasets often have imbalanced classes. Techniques like SMOTE (Synthetic Minority Oversampling Technique) can help address this issue.

In [ ]:
# Generate imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=2, n_classes=2, weights=[0.9, 0.1], random_state=42)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_smote_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# Train models
model_imbalanced = LogisticRegression(random_state=42)
model_smote = LogisticRegression(random_state=42)

model_imbalanced.fit(X_train_scaled, y_train)
model_smote.fit(X_train_smote_scaled, y_train_smote)

# Compare performance
print('Classification Report (Original Imbalanced Data):')
print(classification_report(y_test, model_imbalanced.predict(X_test_scaled)))

print('Classification Report (After SMOTE):')
print(classification_report(y_test, model_smote.predict(X_test_scaled)))

## 5. Model Interpretation

Understanding how features impact predictions is crucial. For logistic regression, we can interpret the coefficients as log-odds.

In [ ]:
# Generate dataset with meaningful feature names
X, y = make_classification(n_samples=1000, n_features=5, random_state=42)
feature_names = ['age', 'income', 'credit_score', 'debt_ratio', 'work_experience']

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Plot feature importance based on coefficients
plt.figure(figsize=(10, 6))
plt.bar(feature_names, model.coef_[0])
plt.title('Feature Importance (Logistic Regression Coefficients)')
plt.xlabel('Features')
plt.ylabel('Coefficient (Log-Odds)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate odds ratios
odds_ratios = np.exp(model.coef_[0])
print('Odds Ratios for each feature:')
for feature, odds in zip(feature_names, odds_ratios):
    print(f'{feature}: {odds:.3f}')